# Multi-Person Detection & Re-Identification - Kaggle Run

Pipeline: YOLOv8 -> DeepSORT -> Re-ID -> Face Verification -> Live Webcam -> Web Console.

**Setup:**
1. Settings -> Internet: On
2. Settings -> Accelerator: GPU T4 x2
3. Add dataset via Kaggle Data -> New Version
4. Run cells top to bottom
5. Last cell prints the URL

**Features:** Registration, video processing, alerts, live webcam, face verification, multi-camera

In [ ]:
# Cell 1: Get project into writable storage
import os, zipfile, shutil, glob

os.chdir("/kaggle/working")
DEST = "/kaggle/working/project"

REQUIRED = ["web/server.py", "models/yolov8s.pt", "reidentification/weights/best_model.pth"]

def verify(p):
    missing = [r for r in REQUIRED if not os.path.exists(os.path.join(p, r))]
    if missing:
        raise SystemExit("Missing: " + ", ".join(missing))
    print("All required files present OK")

if os.path.exists(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST, exist_ok=True)

zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/working/*.zip"))
if zips:
    print("Extracting:", zips[-1])
    with zipfile.ZipFile(zips[-1]) as z:
        z.extractall(DEST)
    verify(DEST)
else:
    candidates = glob.glob("/kaggle/input/datasets/*/*/*/") + glob.glob("/kaggle/input/datasets/*/*/")
    found = None
    for c in candidates:
        if os.path.isfile(os.path.join(c, "web", "server.py")) and os.path.isfile(os.path.join(c, "models", "yolov8s.pt")):
            found = c
            break
    if not found:
        raise SystemExit("No project tree found under /kaggle/input - attach the dataset")
    print("Copying from:", found)
    for entry in sorted(os.listdir(found)):
        s = os.path.join(found, entry)
        d = os.path.join(DEST, entry)
        if os.path.isdir(s):
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    verify(DEST)

print("Project ready:", DEST)
print(os.listdir(DEST))

In [ ]:
# Cell 2: Install dependencies
!apt -qq update > /dev/null 2>&1
!apt -qq install -y ffmpeg > /dev/null 2>&1
%pip install -q ultralytics deep-sort-realtime "fastapi>=0.110" "uvicorn>=0.29" python-multipart insightface onnxruntime gdown scikit-learn matplotlib scipy Pillow tqdm
!apt -qq install -y build-essential cmake > /dev/null 2>&1
%pip install -q face_recognition 2>&1 | tail -5
print("Dependencies installed OK")

In [ ]:
# Cell 3: Download model weights
import gdown
os.chdir(DEST)

if not os.path.exists("models/yolov8s.pt"):
    os.makedirs("models", exist_ok=True)
    gdown.download("https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s.pt", "models/yolov8s.pt", quiet=False)
    print("yolov8s.pt downloaded OK")

if not os.path.exists("reidentification/weights/best_model.pth"):
    os.makedirs("reidentification/weights", exist_ok=True)
    print("WARNING: best_model.pth not found in dataset")

print("Models ready OK")

In [ ]:
# Cell 4: Sanity check
%cd /kaggle/working/project
import sys
sys.path.insert(0, "/kaggle/working/project")

import torch, cv2, ultralytics, fastapi, uvicorn
from detection.detect_module import PersonDetector
from tracking.track_module import PersonTracker
from reidentification.reid_main import ReIDEngine
from reidentification.face_cue import FaceCueExtractor
from reidentification.insight_face import InsightFaceExtractor
from web.server import app
from registration.identity_db import IdentityDatabase
from registration.register_person import register_person

assert torch.cuda.is_available(), "No GPU - Settings -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("All imports verified OK")

In [ ]:
# Cell 5: Check registered people
import sys, os
sys.path.insert(0, "/kaggle/working/project")
os.chdir("/kaggle/working/project")

from registration.identity_db import IdentityDatabase
db = IdentityDatabase()
print(f"Registered persons: {len(db)}")
for name in db:
    print(f"  - {name}")
if len(db) == 0:
    print("No persons registered. Upload photos and register via the web UI or run register.py")

In [ ]:
# Cell 6: Start web server + expose public URL
import os, sys, time, socket, subprocess, re

os.chdir("/kaggle/working/project")

log = open("/kaggle/working/server.log", "w")
server = subprocess.Popen([sys.executable, "web/server.py"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(90):
    s = socket.socket()
    up = s.connect_ex(("127.0.0.1", 8000)) == 0
    s.close()
    if up:
        print("Server is up on port 8000 OK")
        break
    if server.poll() is not None:
        print(open("/kaggle/working/server.log").read())
        raise RuntimeError("Server exited")
    time.sleep(1)
else:
    raise RuntimeError("Server did not open port 8000 in time")

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
t0 = time.time()
while time.time() - t0 < 60:
    line = tunnel.stdout.readline()
    if line:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    else:
        time.sleep(0.25)

if url:
    print()
    print("=" * 62)
    print("OPEN THE WEB CONSOLE HERE:", url)
    print("=" * 62)
    print()
    print("Features:")
    print("  - Upload videos -> Detection + Tracking + Re-ID")
    print("  - Register People -> Face + body enrollment")
    print("  - Live Webcam Recognition -> Real-time boxes + names")
    print("  - Watch-list Alerts -> Flagged persons with photos")
    print("  - Multi-Camera -> Cross-camera identity matching")
else:
    print("Tunnel did not report a URL - re-run this cell")

## Usage notes
- **First run**: ResNet-50 and InsightFace buffalo_s weights download automatically (~1 min extra)
- **Workflow**: Register People -> New Session -> watch progress/results/alerts
- **Live Webcam**: Use the Live Webcam panel in Upload tab
- **URL changes** every re-run
- **Free tier**: ~30 GPU-hours/week, sessions up to ~9h
- **Restart server**: `!pkill -f web/server.py` then re-run Cell 6
- **Check errors**: `!tail -n 40 /kaggle/working/server.log`